# CLEANSING DATA AMAZON PRIME TITLES

Nama : 

    1. Maulita Nur Sejati
    
    2. Ryan Artista Rahman
    
    3. Prita Ayu Meudea
    
    4. Tanzilla Oktavia Widodo
    
    5. Diva Keishya Kirana

### Python Libraries Import

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline
import seaborn as sns

### Load data set

Create a data frame then load the data set. Here the dataset used is amazon_prime_titles. 

In [ ]:
df=pd.read_csv('amazon_prime_titles.csv')

Displays the data set that has been loaded

In [ ]:
df.head()

### Check data condition

In [ ]:
df.shape

9668 : rows, 12 : columns

In [ ]:
df.info()

#### Check Percentage Missing Values

In [ ]:
missing_values = df.isnull().sum()

missing_percentages = (missing_values / len(df)) * 100

missing_info = pd.DataFrame({
    'Column': missing_values.index,
    'Missing Values': missing_values.values,
    'Missing Percentage': missing_percentages.values
})

print(missing_info)

#### Insights and Considerations

Data Completeness:
Columns like 'director,' 'cast,' 'country,' 'date_added,' 'rating,' and others have missing values that need attention.

Data Types:
The 'release_year' column is correctly identified as an integer ('int64'). Other columns, such as 'date_added' and 'duration,' might need conversion to datetime or numeric types if they contain date or duration information.

### Check and remove duplicate rows from the DataFrame 

In [ ]:
df.duplicated().sum()

Count duplicate is 0

### Drop the columns 'date_added' and 'country', from the DataFrame

Because country and date_added column have so many missing data (more than 50%) and there are barely use of them, we will drop them

In [ ]:
df.drop(['date_added','country'], axis = 1, inplace=True)
df.columns

Show dataset info to see if the date and country columns are missing. 
#### Check the others columns missing values.

In [ ]:
df.info()

### Enhance data 
#### rename kolom listed in

In [ ]:
df.rename(columns={'listed_in' : 'categories'},inplace=True)
df.head(2)

#### To solve the data type problems at duration and date added.
- Date added has deleted.
- Then we get attention at the duration columns.

Create a new column named 'duration_num' in the DataFrame,
to return the numerical.

In [ ]:
df['duration_num'] = df['duration'].str.extract('(\d+)').astype(float)

We will split the duration column into duration_movie and duration_season

In [ ]:
df['duration_movie'] = df['duration'].str.extract('(\d+) min', expand=False)

df['duration_season'] = df['duration'].str.extract('(\d+) Seasons', expand=False)

df.drop('duration', axis=1, inplace=True)

df.head()

We will divide the genre into muliple column

In [ ]:
df[['Genre1', 'Genre2', 'Genre3']] = df['categories'].str.split(',', expand=True, n=2)

In [ ]:
df.head()

### Handling Missing Values
Check the others missing values.

In [ ]:
df.isnull().sum()

Then we can fill the missing values with 'unavailable' for columns who are not the categorical and numerical columns.

Fill the rating columns with the mode method.

In [ ]:
df['director'].fillna('Unavailable', inplace=True)
df['cast'].fillna('Unavailable', inplace=True)
df['duration_season'].fillna('Unavailable', inplace=True)
df['duration_movie'].fillna('Unavailable', inplace=True)
df['Genre1'].fillna('Unavailable', inplace=True)
df['Genre2'].fillna('Unavailable', inplace=True)
df['Genre3'].fillna('Unavailable', inplace=True)

df['rating'] = df['rating'].fillna(df['rating'].mode()[0])

In [ ]:
df

#### Check the missing values

In [ ]:
df.isnull().sum()

### Check Outliers
Analyze the distribution of film duration for 'Movie' type films.

The calculated lower and upper bounds can then be used to identify and manage outliers in the film's duration.

In [ ]:
movie_df = df[df['type'] == 'Movie']

q1_movie = movie_df['duration_num'].quantile(0.25)
q3_movie = movie_df['duration_num'].quantile(0.75)

print("Quartil 1 (Q1) untuk Movie:", q1_movie)
print("Quartil 3 (Q3) untuk Movie:", q3_movie)

iqr=q3_movie-q1_movie
print('IQR : ',iqr)

lwr_bound = q1_movie - (1.5 * iqr) #lower bound
upr_bound = q3_movie + (1.5 * iqr) #upper bound

print('Lower Bound : ',lwr_bound)
print('Upper Bound: ',upr_bound)

Arrange the rows of the DataFrame in descending order based on the values in the 'duration_num' column.

In the context of movies, this operation would arrange the movies from the longest to the shortest duration.

In [ ]:
movie_df.sort_values(by='duration_num', ascending=False)

Filter and select rows from the sorted DataFrame movie_df_sorted where the 'duration_num' values fall outside a specific range.

In [ ]:
movie_df_sorted = movie_df.sort_values(by='duration_num', ascending=False)
movie_df_sorted[(movie_df_sorted['duration_num'] > 152) | (movie_df_sorted['duration_num'] < 28)]

### Save Clean Data in CSV format

In [ ]:
df_clean = pd.DataFrame(df)
df_clean.to_csv("amazon_clean.csv")
df_clean